In [9]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

In [10]:
data1 = np.load('dat_split.npz')
X_train, y_train = data1['X_train'], data1['y_train']
X_test_clean, y_test = data1['X_test_clean'], data1['y_test']

In [11]:
n_classes = 40
N_test = len(X_test_clean)
R = 30

In [12]:
best_d = None 
boost_d = 3

In [13]:
e_i = np.zeros((N_test, n_classes))
e_i[np.arange(N_test), y_test] = 1.0

In [14]:
p_tree = np.zeros((R, N_test, n_classes))
p_bag  = np.zeros((R, N_test, n_classes))
p_boost = np.zeros((R, N_test, n_classes))

In [15]:
acc_tree = np.zeros(R)
acc_bag = np.zeros(R)
acc_boost = np.zeros(R)

In [16]:
print(f"Running {R} Bootstrap Repetitions for Bias-Variance Analysis...")

Running 30 Bootstrap Repetitions for Bias-Variance Analysis...


In [17]:
for r in range(R):
   
    boot_idx = []
    for c in range(n_classes):
        c_idx = np.where(y_train == c)[0]
        boot_c = np.random.choice(c_idx, size=len(c_idx), replace=True)
        boot_idx.extend(boot_c)
    
    boot_idx = np.array(boot_idx)
    X_tr, y_tr = X_train[boot_idx], y_train[boot_idx]
    
    # --- B. Single Tree ---
    tree = DecisionTreeClassifier(max_depth=best_d, min_samples_leaf=1).fit(X_tr, y_tr)
    # predict_proba returns the [0, 0, 1, 0...] vector for deep trees
    p_tree[r] = tree.predict_proba(X_test_clean)
    acc_tree[r] = np.mean(tree.predict(X_test_clean) == y_test)
    
    # --- C. Bagging (100 Trees) ---
    bag_preds = np.zeros((N_test, n_classes))
    for b in range(100):
        b_idx = np.random.choice(len(X_tr), size=len(X_tr), replace=True)
        b_tree = DecisionTreeClassifier(max_depth=best_d, min_samples_leaf=1).fit(X_tr[b_idx], y_tr[b_idx])
        
        preds = b_tree.predict(X_test_clean)
        bag_preds[np.arange(N_test), preds] += 1
        
    p_bag[r] = bag_preds / 100.0 
    acc_bag[r] = np.mean(np.argmax(p_bag[r], axis=1) == y_test)
    
    # --- D. Boosting SAMME (100 Rounds) ---
    w = np.ones(len(X_tr)) / len(X_tr)
    boost_preds = np.zeros((N_test, n_classes))
    sum_alpha = 0
    
    for t in range(100):
        bst = DecisionTreeClassifier(max_depth=boost_d, random_state=t).fit(X_tr, y_tr, sample_weight=w)
        preds = bst.predict(X_tr)
        is_wrong = (preds != y_tr)
        err = np.sum(w[is_wrong])
        
        if err >= (1.0 - 1.0/n_classes): break
        if err == 0: err = 1e-10
        
        alpha = np.log((1.0 - err)/err) + np.log(n_classes - 1)
        
       
        test_preds = bst.predict(X_test_clean)
        boost_preds[np.arange(N_test), test_preds] += alpha
        sum_alpha += alpha
        
        w = w * np.exp(alpha * is_wrong)
        w = w / np.sum(w)
        
    p_boost[r] = boost_preds / sum_alpha
    acc_boost[r] = np.mean(np.argmax(p_boost[r], axis=1) == y_test)

In [18]:
def calc_bias_var(p_r):
    p_bar = np.mean(p_r, axis=0) 
    
    bias2 = np.mean(np.sum((p_bar - e_i)**2, axis=1))
    
    var = np.mean(np.sum((p_r - p_bar)**2, axis=2)) 
    
    return bias2, var

In [19]:
bias2_tree, var_tree = calc_bias_var(p_tree)
bias2_bag, var_bag = calc_bias_var(p_bag)
bias2_boost, var_boost = calc_bias_var(p_boost)

In [20]:
results = [
    {"Model": "Single Tree", "Bias^2": bias2_tree, "Variance": var_tree, 
     "Test Acc Mean": np.mean(acc_tree), "Test Acc Std": np.std(acc_tree)},
    
    {"Model": "Bagging (100)", "Bias^2": bias2_bag, "Variance": var_bag, 
     "Test Acc Mean": np.mean(acc_bag), "Test Acc Std": np.std(acc_bag)},
    
    {"Model": "Boosting (100)", "Bias^2": bias2_boost, "Variance": var_boost, 
     "Test Acc Mean": np.mean(acc_boost), "Test Acc Std": np.std(acc_boost)}
]

print("\n--- Q3(a) Bias, Variance, and Test Accuracy ---")
df = pd.DataFrame(results).round(4)
print(df.to_string(index=False))


--- Q3(a) Bias, Variance, and Test Accuracy ---
         Model  Bias^2  Variance  Test Acc Mean  Test Acc Std
   Single Tree  0.3860    0.6523         0.4808        0.0519
 Bagging (100)  0.4764    0.0236         0.7992        0.0319
Boosting (100)  0.9083    0.0147         0.1946        0.0656


In [21]:
data1 = np.load('dat_split.npz')
X_train, y_train = data1['X_train'], data1['y_train']
X_test_clean, y_test = data1['X_test_clean'], data1['y_test']

n_classes = 40
N_test = len(X_test_clean)
R = 30

In [22]:

e_i = np.zeros((N_test, n_classes))
e_i[np.arange(N_test), y_test] = 1.0

In [23]:
depths = [2, 5, None]
depth_names = ["Shallow (2)", "Medium (5)", "Deep (None)"]

In [24]:
p_tree = np.zeros((3, R, N_test, n_classes))
p_bag  = np.zeros((3, R, N_test, n_classes))

In [25]:
for r in range(R):
    if r % 5 == 0 or r == R - 1:
        print(f"--> Processing Repetition {r+1} of {R}...")
        
    # --- A. Stratified Bootstrap ---
    boot_idx = []
    for c in range(n_classes):
        c_idx = np.where(y_train == c)[0]
        boot_idx.extend(np.random.choice(c_idx, size=len(c_idx), replace=True))
    
    boot_idx = np.array(boot_idx)
    X_tr, y_tr = X_train[boot_idx], y_train[boot_idx]
    
    # --- B. Loop through the 3 Depths ---
    for d_idx, d in enumerate(depths):
        
        # 1. Single Tree
        tree = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_tr, y_tr)
        p_tree[d_idx, r] = tree.predict_proba(X_test_clean)
        
        # 2. Bagging (100 Trees)
        bag_preds = np.zeros((N_test, n_classes))
        for b in range(100):
            b_idx = np.random.choice(len(X_tr), size=len(X_tr), replace=True)
            b_tree = DecisionTreeClassifier(max_depth=d, random_state=b).fit(X_tr[b_idx], y_tr[b_idx])
            bag_preds[np.arange(N_test), b_tree.predict(X_test_clean)] += 1
            
        p_bag[d_idx, r] = bag_preds / 100.0

--> Processing Repetition 1 of 30...
--> Processing Repetition 6 of 30...
--> Processing Repetition 11 of 30...
--> Processing Repetition 16 of 30...
--> Processing Repetition 21 of 30...
--> Processing Repetition 26 of 30...
--> Processing Repetition 30 of 30...


In [26]:
results_b = []

for d_idx, name in enumerate(depth_names):
    # Single Tree Metrics
    p_bar_t = np.mean(p_tree[d_idx], axis=0)
    bias2_t = np.mean(np.sum((p_bar_t - e_i)**2, axis=1))
    var_t = np.mean(np.sum((p_tree[d_idx] - p_bar_t)**2, axis=2))
    
    # Bagging Metrics
    p_bar_b = np.mean(p_bag[d_idx], axis=0)
    bias2_b = np.mean(np.sum((p_bar_b - e_i)**2, axis=1))
    var_b = np.mean(np.sum((p_bag[d_idx] - p_bar_b)**2, axis=2))
    
    # Quantify variance removed by bagging
    var_removed = var_t - var_b
    
    results_b.append({
        "Depth": name,
        "Tree Bias^2": bias2_t,
        "Bag Bias^2": bias2_b,
        "Tree Var": var_t,
        "Bag Var": var_b,
        "Var Removed": var_removed
    })

In [27]:
print("\n--- Q3(b) Bias & Variance Across Depths ---")
df_b = pd.DataFrame(results_b).round(4)
print(df_b.to_string(index=False))


--- Q3(b) Bias & Variance Across Depths ---
      Depth  Tree Bias^2  Bag Bias^2  Tree Var  Bag Var  Var Removed
Shallow (2)       0.9281      0.9268    0.0376   0.0104       0.0271
 Medium (5)       0.8709      0.8596    0.0756   0.0143       0.0613
Deep (None)       0.3924      0.4865    0.6259   0.0245       0.6014


Q3.c) No. The deepest tree sees the greatest variance reduction, but a medium-depth tree often sees the largest test accuracy improvement. Our data proves this:

    Bias and Variance: Deep trees have the lowest bias (0.392) but massive variance (0.625). Bagging removes the most variance here (0.601), but bagging cannot fix bias if a shallower tree actually generalizes better.

    Leaf Occupancy: At max_depth=None, leaf occupancy dropped to ~1.0. The trees are just memorizing pure training noise.

    Disagreement: Deep trees disagree the most (43.2%), but this is "noise-driven" disagreement. Averaging 100 models that memorized pure noise doesn't guarantee a better predictor than a medium tree that learned actual facial features ("feature-driven" disagreement).

Model	        Clean_Test_Acc	Masked_Test_Acc	Bias_Squared	Variance
Single_Tree	    0.588	          0.212	         0.3860	        0.6523
Bagging(100)	0.863	          0.600	         0.4764	        0.0236
Boosting(100)	0.300	          0.275	         0.9083	        0.0147


Which model is most stable?
Bagging. While Boosting technically had the lowest variance (0.0147), it completely underfit the data. Bagging is the realistically stable choice, slashing the Single Tree's variance from 0.6523 down to 0.0236 while keeping accuracy high.

Which is most sensitive to the missing region?
The Single Tree. Its accuracy plummeted from 0.588 to 0.212 because if a critical early split falls inside the 16x16 mask, the entire tree fails. Bagging mitigates this (retaining 0.600 accuracy) because its diverse trees split on different, unmasked facial features.

s the model with the lowest variance necessarily the one with the best test accuracy?
No. Boosting achieved the absolute lowest variance (0.0147) but the worst test accuracy (0.300). This perfectly illustrates the Bias-Variance tradeoff: Boosting underfit so severely (highest bias: 0.9083) that it was just consistently wrong. Bagging won because it balanced the low bias of deep trees with artificially reduced variance.

SyntaxError: invalid syntax (2047187929.py, line 1)